<h1>Factory Machine Status</h1>

<hr/>

<p>Name: <strong>Goh Kun Ming</strong><br/></p>
<p>School: <strong>Singapore Polytechnic, School of Computing</strong><br/></p>
<p>Diploma: <strong>Diploma in Applied AI &amp; Analytics</strong><br/></p>
<p>Module: <strong>AI &amp; Machine Learning (ST1511)</strong><br/></p>
<p>Assessment: <strong>CA1 Part A</strong><br/></p>
<p>Academic Period: <strong>AY24/25 Year 1 Semester 2</strong><br/></p>
<p>Lecturer: <strong>Adjunct Lecturer Tai Hock Lin (Andy)</strong><br/></p>

<hr/>

<h1>Notebook Objective</h1>

<hr/>

<p>The objective of this notebook is to train the machine fault prediction model and review the selected probability threshold. I will use a fast sample for notebook demonstration while keeping full training available through the project CLI.</p>

<p>This notebook is part of the broken down notebook workflow. The original CA1 notebook is still maintained as <code>00_original_ca1_submission.ipynb</code>, while this notebook keeps the same report-style explanation and interpretation format in a smaller, easier-to-review file.</p>

<p>Notebook: <strong>03 Training And Thresholding</strong></p>


<hr/>
<h1>1.&nbsp;&nbsp;&nbsp;&nbsp;Importing of Modules</h1>
<hr/>

<p>Within this section, I will import the modules required for sampling data and training the model. The model training logic is stored in the production package so that the notebook demonstrates the workflow without duplicating model code.</p>


<h2>1.1&nbsp;&nbsp;&nbsp;&nbsp;Importing Training Utilities</h2>

<p>The following cell imports pandas for metric display, train-test sampling utilities, project constants, the data loader, and the training function.</p>


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from fault_prediction.config import DEFAULT_DATA_PATH, RANDOM_STATE, TARGET_COLUMN
from fault_prediction.data import load_factory_data, target_distribution
from fault_prediction.models import train_model


<p><strong>Interpretation:</strong> The model will be trained through <code>train_model</code>, which builds the full scikit-learn pipeline, performs preprocessing, trains a soft-voting classifier, and searches for a probability threshold.</p>


<hr/>
<h1>2.&nbsp;&nbsp;&nbsp;&nbsp;Preparing Training Data</h1>
<hr/>

<p>In this section, I will load the dataset and create a smaller stratified sample for notebook execution. This allows the notebook to run quickly while preserving the target class ratio.</p>


<h2>2.1&nbsp;&nbsp;&nbsp;&nbsp;Loading Dataset and Reviewing Target Balance</h2>

<p>Before training the model, I will verify the target distribution again. This is important because the abnormal machine status class is less frequent.</p>


In [ ]:
df = load_factory_data(DEFAULT_DATA_PATH)
target_distribution(df)


<p><strong>Interpretation:</strong> The target distribution confirms that the dataset is imbalanced. Therefore, evaluation should consider balanced accuracy, recall, F1 score, ROC AUC, and average precision.</p>


<hr/>
<h2>2.2&nbsp;&nbsp;&nbsp;&nbsp;Creating a Stratified Notebook Sample</h2>

<p>For the notebook demonstration, I will use a sample of 1,000 rows. Stratified sampling is used so that the sample keeps a similar normal-to-abnormal ratio as the original dataset.</p>


In [ ]:
sample_size = 1000
train_df, _ = train_test_split(
    df,
    train_size=sample_size,
    random_state=RANDOM_STATE,
    stratify=df[TARGET_COLUMN],
)

train_df.shape


<p><strong>Interpretation:</strong> The sampled dataset is smaller than the full dataset, which makes the notebook suitable for quick review. Full training should still be run through the CLI when producing a final model artifact.</p>


<hr/>
<h1>3.&nbsp;&nbsp;&nbsp;&nbsp;Model Training</h1>
<hr/>

<p>In this section, I will train the end-to-end classifier. The training function performs feature engineering, preprocessing, model fitting, validation scoring, and threshold search.</p>


<h2>3.1&nbsp;&nbsp;&nbsp;&nbsp;Training the Voting Classifier</h2>

<p>The fast profile is used here for notebook speed. The standard profile can be used through the CLI for a more complete training run.</p>


In [ ]:
result = train_model(train_df, profile='fast', n_jobs=1)
pd.Series(result.metrics, name='Validation Metric')


<p><strong>Interpretation:</strong> The metrics show how the model performs on the validation split. Because the target is imbalanced, F1 score, recall, balanced accuracy, ROC AUC, and average precision should be reviewed alongside accuracy.</p>


<hr/>
<h2>3.2&nbsp;&nbsp;&nbsp;&nbsp;Threshold Selection</h2>

<p>The model outputs probabilities. A probability threshold is selected to convert probabilities into class predictions. This is useful because the default threshold of 0.5 may not be the best choice for an imbalanced fault detection problem.</p>


In [ ]:
result.threshold_curve.sort_values(
    ['f1', 'recall', 'balanced_accuracy'],
    ascending=False,
).head(10)


<p><strong>Interpretation:</strong> The threshold table shows how model performance changes across different probability thresholds. The selected threshold is stored with the model artifact so prediction uses the same decision rule.</p>


<hr/>
<h2>3.3&nbsp;&nbsp;&nbsp;&nbsp;Reviewing Model Metadata</h2>

<p>The model artifact stores metadata that helps with reproducibility, including the profile, random state, selected threshold, metrics, and final feature names.</p>


In [ ]:
result.artifact['metadata']


<p><strong>Interpretation:</strong> The metadata provides traceability for the trained model. This is important for MLOps because future users need to know how the model was trained and which features it used.</p>


<hr/>
<h1>4.&nbsp;&nbsp;&nbsp;&nbsp;Notebook Summary</h1>
<hr/>

<p>This notebook demonstrates model training and threshold selection using a fast sample. For a final full training run, use the command below from the repository root:</p>

<pre><code>fault-predict train --profile standard</code></pre>
